# 08 — Corrective RAG

**Corrective RAG** extends the self‑reflection idea with a concrete *corrective* step. When retrieved documents are graded as irrelevant, the workflow automatically fetches **web search results** to supplement them.

```
Retrieve Documents
    │
    ▼
Grade Relevance (LLM scores each doc)
    │
    ├── average ≥ 0.5 ──▶ Generate Answer
    │
    └── average < 0.5 ──▶ Correct (Web Search) ──▶ Generate Answer
```

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.langgraph_workflows.corrective_rag_workflow import CorrectiveRAGWorkflow
    print("✓ Imported CorrectiveRAGWorkflow")
except ImportError as e:
    print(f"⚠ Backend import: {e}")
    print("→ Building a local demo")

## 2. Understanding the Flow

The `CorrectiveRAGWorkflow` uses four LangGraph nodes:

| Node | What it does |
|------|-------------|
| `retrieve` | Query the vector store for top‑5 documents |
| `grade` | Score each document's relevance with `RelevanceGrader` |
| `correct` | If avg relevance < 0.5, fetch web results (deduplicated by URL) |
| `generate` | Build context from all docs + corrected docs → generate answer |

## 3. Build Demo Components

In [ ]:
from langchain.embeddings import FakeEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
import asyncio

# Vector store with varied relevance
fake_emb = FakeEmbeddings(size=768)
docs = [
    Document(page_content="Python is a high-level interpreted programming language.",
             metadata={"source": "python_intro.txt"}),
    Document(page_content="Guido van Rossum created Python in 1991 as a hobby project.",
             metadata={"source": "python_history.txt"}),
    Document(page_content="The Mediterranean diet emphasizes fruits, vegetables, and olive oil.",
             metadata={"source": "nutrition.txt"}),
]
vector_store = FAISS.from_documents(docs, fake_emb)

class MockRelevanceGrader:
    async def grade(self, doc_content: str, query: str) -> dict:
        common = len(set(doc_content.lower().split()) & set(query.lower().split()))
        score = min(common / max(len(query.split()), 1), 1.0)
        return {"score": score, "is_relevant": score > 0.3, "explanation": "keyword"}

class MockLLM:
    async def generate_with_context(self, query, context):
        return f"Corrective answer for '{query[:60]}'"
    async def generate(self, prompt):
        return f"Fallback: {prompt[:80]}"

class MockWebRetriever:
    async def retrieve(self, query, top_k=3):
        return [{"content": f"Web supplement for {query}", "source": "https://web.example.com", "relevance_score": 0.8}] * top_k

print("✓ Demo components ready")

## 4. Manual Corrective Flow

Let's step through the logic manually:

In [ ]:
async def corrective_manual(query):
    from backend.adaptive_rag.retrievers.vector_retriever import VectorRetriever
    doc_retriever = VectorRetriever(vector_store)
    web_retriever = MockWebRetriever()
    grader = MockRelevanceGrader()
    
    # Step 1: Retrieve
    print("Step 1 — RETRIEVE")
    docs = await doc_retriever.retrieve(query, top_k=5)
    for d in docs:
        print(f"  {d['content'][:80]}...")
    
    # Step 2: Grade
    print("\nStep 2 — GRADE")
    scores = []
    for d in docs:
        g = await grader.grade(d["content"], query)
        scores.append(g["score"])
        print(f"  Score: {g['score']:.2f} | {'✓' if g['is_relevant'] else '✗'} | {d['content'][:60]}")
    
    avg_score = sum(scores) / len(scores) if scores else 0
    print(f"\n  Average relevance: {avg_score:.2f}")
    
    # Step 3: Correct if needed
    if avg_score < 0.5:
        print("\nStep 3 — CORRECT (web search)")
        web_results = await web_retriever.retrieve(query, top_k=3)
        print(f"  Fetched {len(web_results)} web results to supplement")
        all_docs = docs + web_results
    else:
        print("\nStep 3 — SKIP correction (docs are relevant enough)")
        all_docs = docs
    
    # Step 4: Generate
    print("\nStep 4 — GENERATE")
    context = "\n".join(d.get("content", "") for d in all_docs)
    print(f"  Context: {len(context)} chars from {len(all_docs)} sources")
    print(f"  → Answer generated")

asyncio.run(corrective_manual("Tell me about Python programming"))

## 5. Try a Query Needing Correction

In [ ]:
async def test_correction():
    query = "What are the health benefits of the Mediterranean diet?"
    doc_retriever = VectorRetriever(vector_store)
    docs = await doc_retriever.retrieve(query, top_k=5)
    
    print(f"Query: {query}")
    print(f"Retrieved {len(docs)} document(s)")
    for d in docs:
        overlap = len(set(d['content'].lower().split()) & set(query.lower().split()))
        print(f"  Overlap: {overlap} words | {d['content'][:70]}...")
    
    print("\n→ Since the doc store has poor coverage of 'diet', corrective flow")
    print("  would trigger web search to fill the gap.")

asyncio.run(test_correction())

## 6. CorrectiveRAG vs. Self-RAG

| Aspect | Self-RAG | Corrective RAG |
|--------|----------|----------------|
| **Trigger** | Irrelevant docs → rewrite query | Low avg relevance → web search |
| **Action** | Query rewriting + re‑retrieve | Web search augmentation |
| **Best for** | When better wording helps retrieval | When the doc store lacks the info entirely |
| **Cost** | Multiple LLM calls (rewriting) | One extra web API call |

> **Next:** [09 — LangGraph Workflow](./09_LangGraph_Workflow.ipynb) — the complete Adaptive RAG state machine